# DataForge pipeline worker on Colab's GPU (WiLoR + depth model)

For **testing**. The app runs on your laptop (`./run-web-for-colab.sh`); this notebook is its **pipeline worker**: it takes the app's jobs, reads the clips, runs the pipeline on the GPU and writes the results back — the admin panel shows them as they finish. The same worker as the DGX, with this repository's pipeline (WiLoR for the hand, the MoGe-2 depth model for the distances; GPU only).

```
phone / web ──upload──▶ app on the laptop ──job──▶ Redis (laptop)
                                                     │  Tailscale
                     this notebook (Colab GPU) ◀─────┘
                     clip from the laptop's storage (S3) → pipeline → results back
```

Colab joins your **Tailscale** network; nothing on the laptop is opened to the internet.

### Before you start
1. **Runtime → Change runtime type → T4 GPU.**
2. On the laptop: `./run-web-for-colab.sh` — it prints the laptop's Tailscale address and the two passwords below.
3. **Colab secrets** (🔑 on the left; give this notebook access):
   - `TS_AUTHKEY` — Tailscale admin console → Settings → Keys → *Generate auth key*, **Ephemeral** on (Colab leaves the network when it stops)
   - `REDIS_PASSWORD`, `S3_SECRET_KEY` — from the laptop script
   - optional, for the AI check (queue `check`): `LLM_API_BASE`, `LLM_API_KEY`, `LLM_MODEL` — a model **this Colab can reach**; without one, clips wait at the check
   - optional, to publish datasets: `HF_TOKEN`

Then run the cells **top to bottom**. Cell 6 keeps running: that is the worker.

In [ ]:
#@title 1 · Settings
LAPTOP_IP = "100.117.13.88"  #@param {type:"string"}
# check: the AI check (needs a model, below) · track: hand tracking (the GPU) · build: datasets
QUEUES = "check,track,build"  #@param {type:"string"}
CODE_REPO = 'https://github.com/Aravind-0808/pipeline-colab-tmp.git'   # temporary, public; branch main

import subprocess, sys
assert sys.version_info >= (3, 12), f"The pipeline needs Python 3.12+, this runtime has {sys.version.split()[0]}"
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
assert gpu.returncode == 0, 'No GPU: Runtime → Change runtime type → T4 GPU, then run again from cell 1'
print('GPU:', gpu.stdout.strip())

from google.colab import userdata
def secret(name, default=None):
    try:
        return userdata.get(name)
    except Exception:
        if default is None:
            raise RuntimeError(f"Add the Colab secret {name} (🔑 on the left) and give this notebook access")
        return default
for name in ('TS_AUTHKEY', 'REDIS_PASSWORD', 'S3_SECRET_KEY'):
    secret(name)
print('secrets OK | AI check model:', 'set' if secret('LLM_API_BASE', '') else 'none (clips will wait at the check)')

In [ ]:
#@title 2 · Code and packages (5–10 min the first time; re-run to pick up a newer push)
import os
if not os.path.isdir('/content/pipeline/.git'):
    !git clone -q {CODE_REPO} /content/pipeline
else:
    !git -C /content/pipeline pull -q
%cd /content/pipeline
!git log --oneline -1
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y ffmpeg libegl1 libgles2 socat > /dev/null 2>&1
# PyTorch pinned to the CUDA 12.6 build Colab's driver runs: a newer lerobot or
# torchcodec otherwise pulls PyTorch 2.11 for CUDA 13, whose cuSOLVER fails on
# Colab's driver (the depth model's lens estimate: CUSOLVER_STATUS_INTERNAL_ERROR).
# lerobot pinned too: without it pip walks back through releases that lack [dataset].
TORCH = ['torch==2.7.1', 'torchvision==0.22.1']
!pip install -q --index-url https://download.pytorch.org/whl/cu126 {' '.join(TORCH)}
!pip install -q -e ".[ml]" "lerobot[dataset]==0.6.1" {' '.join(TORCH)} --extra-index-url https://download.pytorch.org/whl/cu126
!python -c "import torch; print('PyTorch', torch.__version__, '| CUDA', torch.version.cuda); assert torch.__version__.startswith('2.7.1')"
!pip -q uninstall -y opencv-contrib-python opencv-python opencv-python-headless
!pip -q install "opencv-contrib-python-headless==5.0.0.93"
!sh scripts/install_perception.sh depth
!python scripts/download_retarget_weights.py --dir ./models
!python scripts/download_perception_weights.py --dir ./models depth
!python -c "import cv2, mediapipe, scipy, moge, dfpipeline.worker; print('imports OK')"

In [ ]:
#@title 3 · WiLoR
# chumpy: a small stand-in, written below. The real chumpy no longer builds on
# Python 3.12+ / numpy 2, and WiLoR needs it only to read MANO's pickle (two classes).
import os, textwrap
d = '/content/chumpy_shim/chumpy'; os.makedirs(d, exist_ok=True)
open(f'{d}/__init__.py', 'w').write('from chumpy.ch import Ch\n')
open(f'{d}/ch.py', 'w').write(textwrap.dedent('''
    import numpy as np
    class Ch:
        # smplx reads .shape and slices before it converts with .r, so this
        # behaves as the array it holds.
        def __setstate__(self, state): self.__dict__.update(state)
        @property
        def r(self): return np.asarray(self.x)
        def __array__(self, dtype=None, copy=None):
            a = np.asarray(self.r)
            return a.astype(dtype) if dtype is not None else a
        @property
        def shape(self): return np.asarray(self.r).shape
        @property
        def ndim(self): return np.asarray(self.r).ndim
        @property
        def dtype(self): return np.asarray(self.r).dtype
        def __len__(self): return len(np.asarray(self.r))
        def __getitem__(self, i): return np.asarray(self.r)[i]
'''))
open(f'{d}/reordering.py', 'w').write(textwrap.dedent('''
    import numpy as np
    from chumpy.ch import Ch
    class Select(Ch):
        @property
        def r(self):
            v = np.asarray(self.a.r).ravel()[np.asarray(self.idxs)]
            s = getattr(self, "preferred_shape", None)
            return v.reshape(s) if s is not None else v
'''))
open('/content/chumpy_shim/pyproject.toml', 'w').write(textwrap.dedent('''
    [project]
    name = "chumpy"
    version = "0.0.0"
    dependencies = ["numpy"]
    [build-system]
    requires = ["setuptools>=61"]
    build-backend = "setuptools.build_meta"
    [tool.setuptools]
    packages = ["chumpy"]
'''))
!pip install -q --force-reinstall --no-deps /content/chumpy_shim
!pip install -q --no-deps "wilor-mini @ git+https://github.com/warmshao/WiLoR-mini.git@ebec42f94c389070cdd7dda6fd1bf0b4a659c960" "smplx==0.1.28" "ultralytics==8.1.34"
# --no-deps: thop and friends would otherwise upgrade PyTorch and pull a second
# OpenCV, which overwrites the headless one (cv2 then has no VideoCapture).
!pip install -q --no-deps timm einops roma py-cpuinfo thop
!pip install -q scikit-image torch==2.7.1 torchvision==0.22.1 --extra-index-url https://download.pytorch.org/whl/cu126
# Whatever came in, one OpenCV again: the headless contrib build MediaPipe takes.
!pip -q uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
!pip -q install --force-reinstall --no-deps "opencv-contrib-python-headless==5.0.0.93"
!python -c "import cv2; assert hasattr(cv2, 'VideoCapture'); print('OpenCV OK', cv2.__version__)"
!python scripts/download_perception_weights.py --dir ./models wilor
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

In [ ]:
#@title 4 · Join Tailscale and reach the laptop's Redis and storage on localhost
import os, subprocess, time
!curl -fsSL https://tailscale.com/install.sh | sh > /dev/null
SOCK = '/tmp/tailscaled.sock'
# Userspace networking: no network device of its own, which is all Colab allows.
if subprocess.run(['tailscale', f'--socket={SOCK}', 'status'], capture_output=True).returncode != 0:
    subprocess.Popen(['tailscaled', '--tun=userspace-networking', '--state=mem:', f'--socket={SOCK}'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    subprocess.run(['tailscale', f'--socket={SOCK}', 'up', f'--authkey={secret("TS_AUTHKEY")}',
                    '--hostname=colab-gpu-worker', '--accept-dns=false'], check=True)
# In userspace mode programs cannot open Tailscale addresses themselves, so each
# port is relayed: localhost:PORT → `tailscale nc` → laptop:PORT. The laptop's
# Redis for this is on 16379 (run-web-for-colab.sh), its storage on 19000.
!pkill -x socat || true
for local, remote in ((6379, 16379), (19000, 19000)):
    subprocess.Popen(['socat', f'TCP-LISTEN:{local},fork,reuseaddr,bind=127.0.0.1',
                      f'EXEC:tailscale --socket={SOCK} nc {LAPTOP_IP} {remote}'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
print(subprocess.run(['tailscale', f'--socket={SOCK}', 'ping', '-c', '1', LAPTOP_IP],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 5 · Settings for the worker, and every connection checked
import os
WORKER_ENV = {
    **os.environ,
    'REDIS_URL': f"redis://:{secret('REDIS_PASSWORD')}@127.0.0.1:6379/0",
    'S3_ENDPOINT': 'http://127.0.0.1:19000',
    'S3_ACCESS_KEY': 'pipeline',
    'S3_SECRET_KEY': secret('S3_SECRET_KEY'),
    'S3_BUCKET': 'dataforge',
    'LLM_API_BASE': secret('LLM_API_BASE', ''),
    'LLM_API_KEY': secret('LLM_API_KEY', 'EMPTY'),
    'LLM_MODEL': secret('LLM_MODEL', ''),
    'HF_TOKEN': secret('HF_TOKEN', ''),
    'PIPELINE_HOURS': '',                 # always on while testing (the default is the DGX's nights)
    'PIPELINE_TZ': 'Asia/Kolkata',
    'NODE_NAME': 'colab',
    'EXPORT_DATA_DIR': '/content/_datasets',
    'RETARGET_MODELS_DIR': '/content/pipeline/models',
    'GPU_MEMORY_FRACTION': '0.9',         # the T4 is the worker's alone
    'TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD': '1',
}
WORKER_ENV = {k: v for k, v in WORKER_ENV.items() if v != ''} | {'PIPELINE_HOURS': ''}
CHECK = r"""
import redis
from dfpipeline.config import settings
from dfpipeline.storage import get_storage
print('Redis  :', 'OK' if redis.Redis.from_url(settings.redis_url, socket_timeout=10).ping() else 'no answer')
s = get_storage(); s.ensure_bucket()
print('storage:', 'OK —', sum(1 for _ in s.client.list_objects(s.bucket, recursive=False)), 'folders in', s.bucket)
"""
run = subprocess.run([sys.executable, '-c', CHECK], cwd='/content/pipeline', env=WORKER_ENV, capture_output=True, text=True)
print(run.stdout.strip() or run.stderr.strip()[-800:])
!cd /content/pipeline && RETARGET_MODELS_DIR=./models GPU_MEMORY_FRACTION=0.9 python scripts/gpu_smoke.py

In [ ]:
#@title 6 · The worker (keeps running — ■ stops it)
# Two Celery workers, as on the DGX: hand tracking one clip at a time (the GPU's
# models stay loaded between clips), and the checks and dataset builds beside it.
# Upload a recording from the phone and watch the jobs arrive here; results show
# in the admin panel as they finish.
import signal, subprocess, sys, threading, time
queues = [q.strip() for q in QUEUES.split(',') if q.strip()]
groups = []
if 'track' in queues:
    groups.append(('track', ['track'], ['--max-memory-per-child=8192000']))
rest = [q for q in queues if q != 'track']
if rest:
    groups.append(('other', rest, []))
procs = []
for name, qs, extra in groups:
    cmd = [sys.executable, '-m', 'celery', '-A', 'dfpipeline.worker', 'worker', '-Q', ','.join(qs), '-c', '1',
           '-n', f'colab-{name}@%h', '--loglevel=INFO', *extra]
    p = subprocess.Popen(cmd, cwd='/content/pipeline', env=WORKER_ENV, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    procs.append(p)
    threading.Thread(target=lambda p=p, n=name: [print(f'{n:5} | {l}', end='') for l in p.stdout],
                     daemon=True).start()
print('workers:', ', '.join(f"{n} → {','.join(q)}" for n, q, _ in groups))
try:
    while all(p.poll() is None for p in procs):
        time.sleep(5)
    print('a worker stopped:', [p.returncode for p in procs])
except KeyboardInterrupt:
    print('stopping — the job in hand is finished first (■ again to stop at once)')
    for p in procs:
        p.send_signal(signal.SIGTERM)
    for p in procs:
        p.wait()
    print('stopped')

### When you are done
- Stop cell 6 (■), then **Runtime → Disconnect and delete runtime** — Colab leaves your Tailscale network (the key is ephemeral).
- Work that was running when Colab stopped is not lost: it was never acknowledged, so Redis hands it out again to the next worker.
- Stop the laptop script with Ctrl+C.